----
## Shreya Kusumkar 
## PRN : 260840128044
## PML Assignment : 3
----

In [35]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV,KFold,RandomizedSearchCV
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import os

import warnings
warnings.filterwarnings("ignore")

In [36]:
from ISLP import load_data
Bikeshare = load_data('Bikeshare')
Bikeshare.columns

Index(['season', 'mnth', 'day', 'hr', 'holiday', 'weekday', 'workingday',
       'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual',
       'registered', 'bikers'],
      dtype='object')

In [37]:
Bikeshare

,season,mnth,day,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,bikers
0,1,Jan,1,0,0,6,0,clear,0.24,0.2879,0.81,0.0000,3,13,16
1,1,Jan,1,1,0,6,0,clear,0.22,0.2727,0.80,0.0000,8,32,40
2,1,Jan,1,2,0,6,0,clear,0.22,0.2727,0.80,0.0000,5,27,32
3,1,Jan,1,3,0,6,0,clear,0.24,0.2879,0.75,0.0000,3,10,13
4,1,Jan,1,4,0,6,0,clear,0.24,0.2879,0.75,0.0000,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8640,1,Dec,365,19,0,6,0,clear,0.42,0.4242,0.54,0.2239,19,73,92
8641,1,Dec,365,20,0,6,0,clear,0.42,0.4242,0.54,0.2239,8,63,71
8642,1,Dec,365,21,0,6,0,clear,0.40,0.4091,0.58,0.1940,2,50,52
8643,1,Dec,365,22,0,6,0,clear,0.38,0.3939,0.62,0.1343,2,36,38


In [38]:
Bikeshare['season'].value_counts()

season
3    2240
2    2203
4    2134
1    2068
Name: count, dtype: int64

In [39]:
X,y = Bikeshare.drop('bikers',axis=1) ,Bikeshare['bikers']

In [40]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)

In [41]:
en = ElasticNet(random_state=26)
knn = KNeighborsRegressor()
gbm = GradientBoostingRegressor(random_state=26)
xgb = XGBRegressor()
lbm = LGBMRegressor()
cat = CatBoostRegressor()
rf = RandomForestRegressor(random_state=26)
kfold = KFold(shuffle=True,random_state=26,n_splits=5)

In [42]:
pipe_en = Pipeline([('OHE',ct),('EN',en)])
pipe_rfr = Pipeline([('OHE',ct),('RFR',rf)])
pipe_xgb = Pipeline([('OHE',ct),('XGB',xgb)])
pipe_gbr = Pipeline([('OHE',ct),('GBR',gbm)])
pipe_cat = Pipeline([('OHE',ct),('CAT',cat)])
pipe_lgb = Pipeline([('OHE',ct),('LGBM',lbm)])

----
# ElasticNet
----

In [43]:
pipe_en.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE4050>)],
                     verbose_feature_names_out=False)),
  ('EN', ElasticNet(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'OHE': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
 

### Gridsearch CV

In [44]:
params = { 'EN__alpha': [.1,.25,.7,1,1.5,2,5,10,50],
           'EN__l1_ratio': [.1,.25,.5,.75,1] }
gcv_en = GridSearchCV( pipe_en, param_grid= params, cv = kfold, n_jobs= -1)
gcv_en.fit(X,y)
print("Best param : ", gcv_en.best_params_)
print("Best score : ", gcv_en.best_score_)

Best param :  {'EN__alpha': 0.1, 'EN__l1_ratio': 1}
Best score :  0.9999999996114909


### Randomized Grid Search CV

In [45]:
params = { 'EN__alpha': [.1,.25,.7,1,1.5,2,5,10,50],
           'EN__l1_ratio': [.1,.25,.5,.75,1] }
gcv_en = RandomizedSearchCV( pipe_en,n_iter=5, param_distributions= params, cv = kfold, n_jobs= -1)
gcv_en.fit(X,y)
print("Best param : ", gcv_en.best_params_)
print("Best score : ", gcv_en.best_score_)

Best param :  {'EN__l1_ratio': 1, 'EN__alpha': 0.1}
Best score :  0.9999999996114909


-----
# Random Forest
-----

In [46]:
pipe_rfr.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE4050>)],
                     verbose_feature_names_out=False)),
  ('RFR', RandomForestRegressor(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'OHE': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000021AB

In [47]:
params = {
    'RFR__max_features': [2,3,4,5],
    'RFR__n_estimators': [10,25,50,100],
    'RFR__max_depth': [None,1,2,3,4,5,6,7]
}
gcv_rfr = GridSearchCV(pipe_rfr, param_grid= params, cv = kfold, n_jobs=-1)
gcv_rfr.fit(X,y)
print("Best param : ", gcv_rfr.best_params_)
print("Best score : ", gcv_rfr.best_score_)

Best param :  {'RFR__max_depth': None, 'RFR__max_features': 5, 'RFR__n_estimators': 100}
Best score :  0.9988735924728328


### Randomized Grid Search CV

In [48]:
params = {
    'RFR__max_features': [2,3,4,5],
    'RFR__n_estimators': [10,25,50,100],
    'RFR__max_depth': [None,1,2,3,4,5,6,7]
}
gcv_rfr = RandomizedSearchCV(pipe_rfr,n_iter=10 ,param_distributions= params, cv = kfold, n_jobs=-1)
gcv_rfr.fit(X,y)
print("Best param : ", gcv_rfr.best_params_)
print("Best score : ", gcv_rfr.best_score_)

Best param :  {'RFR__n_estimators': 25, 'RFR__max_features': 5, 'RFR__max_depth': None}
Best score :  0.9986622018871134


# Gradient Boosting

In [49]:
pipe_gbr.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE4050>)],
                     verbose_feature_names_out=False)),
  ('GBR', GradientBoostingRegressor(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'OHE': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x00000

In [50]:
params = {
    'GBR__learning_rate' : np.linspace(0.01,0.9,10),
    'GBR__max_depth' : [None,2,3,4],
    'GBR__n_estimators' : [25,50,100]
}
gcv_gbr = GridSearchCV(pipe_gbr, param_grid= params, cv = kfold, n_jobs=-1)
gcv_gbr.fit(X,y)
print("Best param : ", gcv_gbr.best_params_)
print("Best score : ", gcv_gbr.best_score_)

Best param :  {'GBR__learning_rate': np.float64(0.10888888888888888), 'GBR__max_depth': 4, 'GBR__n_estimators': 100}
Best score :  0.9995614049446825


### Randomized Grid Search CV

In [51]:
params = {
    'GBR__learning_rate' : np.linspace(0.01,0.9,10),
    'GBR__max_depth' : [None,2,3,4],
    'GBR__n_estimators' : [25,50,100]
}
gcv_gbr = RandomizedSearchCV(pipe_gbr, n_iter=10, param_distributions=params, cv = kfold, n_jobs=-1)
gcv_gbr.fit(X,y)
print("Best param : ", gcv_gbr.best_params_)
print("Best score : ", gcv_gbr.best_score_)

Best param :  {'GBR__n_estimators': 100, 'GBR__max_depth': 3, 'GBR__learning_rate': np.float64(0.8011111111111111)}
Best score :  0.9993340526107236


# XGBRegressor

In [52]:
pipe_xgb.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE4050>)],
                     verbose_feature_names_out=False)),
  ('XGB',
   XGBRegressor(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=True, eval_metric=None, feature_types=None,
                feature_weights=None, gamma=None, grow_policy=None,
                importance_type=None,

In [53]:
params = {
    'XGB__learning_rate' : np.linspace(0.01,0.9,10),
    'XGB__max_depth' : [None,2,3,4],
    'XGB__n_estimators' : [25,50,100]
}
gcv_xgb = GridSearchCV(pipe_xgb, param_grid= params, cv = kfold, n_jobs=-1)
gcv_xgb.fit(X,y)
print("Best param : ", gcv_xgb.best_params_)
print("Best score : ", gcv_xgb.best_score_)

Best param :  {'XGB__learning_rate': np.float64(0.10888888888888888), 'XGB__max_depth': None, 'XGB__n_estimators': 100}
Best score :  0.9996362328529358


### Randomized Grid Search CV

In [54]:
params = {
    'XGB__learning_rate' : np.linspace(0.01,0.9,10),
    'XGB__max_depth' : [None,2,3,4],
    'XGB__n_estimators' : [25,50,100]
}
gcv_xgb = RandomizedSearchCV(pipe_xgb,n_iter=10, param_distributions= params, cv = kfold, n_jobs=-1)
gcv_xgb.fit(X,y)
print("Best param : ", gcv_xgb.best_params_)
print("Best score : ", gcv_xgb.best_score_)

Best param :  {'XGB__n_estimators': 100, 'XGB__max_depth': None, 'XGB__learning_rate': np.float64(0.10888888888888888)}
Best score :  0.9996362328529358


# CatBoost

In [55]:
pipe_cat.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE4050>)],
                     verbose_feature_names_out=False)),
  ('CAT', CatBoostRegressor(loss_function='RMSE'))],
 'transform_input': None,
 'verbose': False,
 'OHE': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000021A

In [56]:
params = {
    'CAT__learning_rate' : np.linspace(0.01,0.9,10),
    'CAT__depth' : [None,2,3,4],
    'CAT__iterations' : [25,50,100]
}
gcv_cat = GridSearchCV(pipe_cat, param_grid= params, cv = kfold, n_jobs=-1)
gcv_cat.fit(X,y)
print("Best param : ", gcv_cat.best_params_)
print("Best score : ", gcv_cat.best_score_)

0:	learn: 49.9314983	total: 3.01ms	remaining: 298ms
1:	learn: 35.1145152	total: 5.91ms	remaining: 290ms
2:	learn: 29.3144562	total: 8.19ms	remaining: 265ms
3:	learn: 24.0045182	total: 10.5ms	remaining: 253ms
4:	learn: 20.2181933	total: 12.9ms	remaining: 245ms
5:	learn: 16.9271531	total: 15.2ms	remaining: 238ms
6:	learn: 16.1403215	total: 17.3ms	remaining: 230ms
7:	learn: 14.8839690	total: 19.4ms	remaining: 223ms
8:	learn: 13.7660938	total: 21.8ms	remaining: 220ms
9:	learn: 12.9476171	total: 24ms	remaining: 216ms
10:	learn: 12.0516269	total: 26.2ms	remaining: 212ms
11:	learn: 11.2740405	total: 28.3ms	remaining: 208ms
12:	learn: 10.7878902	total: 30.9ms	remaining: 207ms
13:	learn: 10.3036050	total: 32.8ms	remaining: 201ms
14:	learn: 9.9696897	total: 34.9ms	remaining: 198ms
15:	learn: 9.3069607	total: 36.8ms	remaining: 193ms
16:	learn: 8.7366706	total: 38.7ms	remaining: 189ms
17:	learn: 8.1278476	total: 40.6ms	remaining: 185ms
18:	learn: 7.7200184	total: 42.6ms	remaining: 182ms
19:	learn:

### Random Grid Search CV

In [57]:
params = {
    'CAT__learning_rate' : np.linspace(0.01,0.9,10),
    'CAT__depth' : [None,2,3,4],
    'CAT__iterations' : [25,50,100]
}
gcv_cat = RandomizedSearchCV(pipe_cat,n_iter=10, param_distributions= params, cv = kfold,n_jobs=-1)
gcv_cat.fit(X,y)
print("Best param : ", gcv_cat.best_params_)
print("Best score : ", gcv_cat.best_score_)

0:	learn: 54.4066893	total: 2.57ms	remaining: 254ms
1:	learn: 34.5004418	total: 4.83ms	remaining: 237ms
2:	learn: 28.7347956	total: 7.02ms	remaining: 227ms
3:	learn: 25.0374925	total: 9.23ms	remaining: 221ms
4:	learn: 21.6168561	total: 11.3ms	remaining: 214ms
5:	learn: 19.3658543	total: 13.6ms	remaining: 213ms
6:	learn: 17.1141253	total: 15.7ms	remaining: 208ms
7:	learn: 16.0242997	total: 17.6ms	remaining: 202ms
8:	learn: 14.6396593	total: 19.6ms	remaining: 198ms
9:	learn: 13.6302288	total: 21.5ms	remaining: 194ms
10:	learn: 12.7448338	total: 23.5ms	remaining: 190ms
11:	learn: 12.5130016	total: 25.2ms	remaining: 185ms
12:	learn: 11.9200124	total: 27.1ms	remaining: 181ms
13:	learn: 11.6030785	total: 29ms	remaining: 178ms
14:	learn: 11.1621426	total: 30.9ms	remaining: 175ms
15:	learn: 10.2267885	total: 32.9ms	remaining: 173ms
16:	learn: 9.8530981	total: 34.8ms	remaining: 170ms
17:	learn: 9.2009156	total: 36.7ms	remaining: 167ms
18:	learn: 8.8318234	total: 38.7ms	remaining: 165ms
19:	lear

# LightGBM

In [58]:
pipe_lgb.get_params()

{'memory': None,
 'steps': [('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE4050>)],
                     verbose_feature_names_out=False)),
  ('LGBM', LGBMRegressor())],
 'transform_input': None,
 'verbose': False,
 'OHE': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000021ABCAE7A10>),
           

In [59]:
params = {
    'LGBM__learning_rate' : np.linspace(0.01,0.9,10),
    'LGBM__max_depth' : [None,2,3,4],
    'LGBM__n_estimators' : [25,50,100]
}
gcv_lgb = GridSearchCV(pipe_lgb, param_grid= params, cv = kfold, n_jobs=-1)
gcv_lgb.fit(X,y)
print("Best param : ", gcv_lgb.best_params_)
print("Best score : ", gcv_lgb.best_score_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000453 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 938
[LightGBM] [Info] Number of data points in the train set: 8645, number of used features: 11
[LightGBM] [Info] Start training from score 143.794448
Best param :  {'LGBM__learning_rate': np.float64(0.10888888888888888), 'LGBM__max_depth': None, 'LGBM__n_estimators': 100}
Best score :  0.9995433869783042


### Random Grid Search CV

In [60]:
params = {
    'LGBM__learning_rate' : np.linspace(0.01,0.9,10),
    'LGBM__max_depth' : [None,2,3,4],
    'LGBM__n_estimators' : [25,50,100]
}
gcv_lgb = RandomizedSearchCV(pipe_lgb,n_iter=10, param_distributions= params, cv = kfold,n_jobs=-1)
gcv_lgb.fit(X,y)
print("Best param : ", gcv_lgb.best_params_)
print("Best score : ", gcv_lgb.best_score_)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000212 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 938
[LightGBM] [Info] Number of data points in the train set: 8645, number of used features: 11
[LightGBM] [Info] Start training from score 143.794448
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: